# 07회차 실습 노트북 — 데이터 분석 모델의 평가

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/07_model_evaluation.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/07_model_evaluation.html](https://adminmrobo.github.io/fintech_data_analysis/07_model_evaluation.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 여행자 보험 청구 데이터: 청구가 1.5%뿐

In [2]:
t = load('Revolution/travel insurance.csv')
t['y'] = (t.Claim == 'Yes').astype(int)
print(t.shape, '청구 비율', round(t.y.mean(), 4))

(63326, 12) 청구 비율 0.0146


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, recall_score, accuracy_score
X = pd.get_dummies(t[['Agency', 'Agency Type', 'Distribution Channel', 'Product Name', 'Duration',
                      'Net Sales', 'Commision (in value)', 'Age']], drop_first=True).astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, t.y, test_size=.25, random_state=RS, stratify=t.y)
s = StandardScaler().fit(Xtr)
pr = LogisticRegression(max_iter=3000).fit(s.transform(Xtr), ytr).predict_proba(s.transform(Xte))[:, 1]
print('AUC', round(roc_auc_score(yte, pr), 3))

AUC 0.852


## 2. 대결 재현: "모두 청구 없음" vs 로지스틱(임계값 0.02)

In [4]:
for name, p in [('모두 청구 없음', np.zeros(len(yte), int)), ('로지스틱 0.02', (pr >= .02).astype(int))]:
    tn, fp, fn, tp = confusion_matrix(yte, p).ravel()
    print(f'{name:10s} 정확도 {accuracy_score(yte, p):.3f}  재현율 {recall_score(yte, p, zero_division=0):.3f}  TP={tp} FP={fp} FN={fn} TN={tn}')

모두 청구 없음   정확도 0.985  재현율 0.000  TP=0 FP=0 FN=232 TN=15600
로지스틱 0.02  정확도 0.854  재현율 0.668  TP=155 FP=2227 FN=77 TN=13373


## 3. 임계값을 바꾸면

In [5]:
for th in [0.01, 0.02, 0.05, 0.1, 0.3, 0.5]:
    tn, fp, fn, tp = confusion_matrix(yte, (pr >= th).astype(int)).ravel()
    print(f'임계값 {th:<5} 잡은 청구 {tp:3d} / 놓친 청구 {fn:3d} / 잘못 경보 {fp:5d}')

임계값 0.01  잡은 청구 196 / 놓친 청구  36 / 잘못 경보  4421
임계값 0.02  잡은 청구 155 / 놓친 청구  77 / 잘못 경보  2227
임계값 0.05  잡은 청구 128 / 놓친 청구 104 / 잘못 경보  1269
임계값 0.1   잡은 청구  32 / 놓친 청구 200 / 잘못 경보   234
임계값 0.3   잡은 청구   0 / 놓친 청구 232 / 잘못 경보     6
임계값 0.5   잡은 청구   0 / 놓친 청구 232 / 잘못 경보     1


**AI 판단 연습** — 원고의 "정확도 77% → 97%로 개선" 같은 문장을 보면, 같은 데이터에서 "모두 아니오"의 정확도부터 계산해 보세요.